# <center> Titanic Classification </center>
----

data source: https://www.kaggle.com/competitions/titanic/data?select=train.csv



In [294]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MaxAbsScaler, StandardScaler


In [303]:
titanic = pd.read_csv("titanic_train.csv")
temp_cols = titanic[["Pclass", "Survived"]]
titanic = titanic.drop(columns=["Pclass", "Survived"])
titanic[["Pclass", "Survived"]] = temp_cols


################################################# Data Clean up ####################################################
##### removed rows with NaN age
titanic = titanic[~titanic["Age"].isna()]

##### removed rows with NaN Embarked
titanic = titanic[~titanic["Embarked"].isna()]

##### removed rows with Fare == 0
# titanic = titanic[titanic["Fare"]>0]

##### reset index
titanic = titanic.reset_index(drop=True)


################################################# Feature Engineering ##############################################

##### split first and last name
titanic["FirstName"] = [titanic.loc[val, "Name"].split(",")[1] for val in range(len(titanic))]
titanic["LastName"] = [titanic.loc[val, "Name"].split(",")[0] for val in range(len(titanic))]

##### get salutation
titanic["Salutation"] = [titanic.loc[val, "FirstName"].split()[0] for val in range(len(titanic))]

##### determine family size
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"]

##### Pclass-Age interaction
titanic["Pclass*Age"] = titanic["Pclass"]*titanic["Age"]


titanic

,PassengerId,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass,Survived,FirstName,LastName,Salutation,FamilySize,Pclass*Age
0,1,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,3,0,Mr. Owen Harris,Braund,Mr.,1,66.0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1,1,Mrs. John Bradley (Florence Briggs Thayer),Cumings,Mrs.,1,38.0
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,3,1,Miss. Laina,Heikkinen,Miss.,0,78.0
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1,1,Mrs. Jacques Heath (Lily May Peel),Futrelle,Mrs.,1,35.0
4,5,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,3,0,Mr. William Henry,Allen,Mr.,0,105.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,886,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,NaN,Q,3,0,Mrs. William (Margaret Norton),Rice,Mrs.,5,117.0
708,887,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,2,0,Rev. Juozas,Montvila,Rev.,0,54.0
709,888,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,1,1,Miss. Margaret Edith,Graham,Miss.,0,19.0
710,890,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,1,1,Mr. Karl Howell,Behr,Mr.,0,26.0


In [273]:
# titanic.groupby(["Salutation"]).count()
# titanic[titanic["Salutation"].isin(["Mme.", "Mlle.", "Don.", "the"])]
last_name_unique

['Davis',
 'Adams',
 'Harmer',
 'Foreman',
 'Gavey',
 'Bailey',
 'Newell',
 'Van Impe',
 'Cavendish',
 'Alhomaki',
 'Persson',
 'Daniel',
 'Lobb',
 'Peuchen',
 'Strandberg',
 'Theobald',
 'Coutts',
 'Vander Cruyssen',
 'Lang',
 'Nirva',
 'Widegren',
 'Guggenheim',
 'Perreault',
 'Davidson',
 'Hansen',
 'Hakkarainen',
 'Fortune',
 'Carr',
 'Charters',
 'Markun',
 'Appleton',
 'Chip',
 'Morley',
 'LeRoy',
 'Danbom',
 'Barkworth',
 'Ringhini',
 'Calderhead',
 'Jonkoff',
 'Nicola-Yarred',
 'Zabour',
 'Eustis',
 'Carter',
 'Ponesell',
 'Windelov',
 'Kirkland',
 'Saad',
 'Fynney',
 'Asplund',
 'Jussila',
 'Maenpaa',
 'Stankovic',
 'Larsson',
 'Romaine',
 'Hoyt',
 'Serepeca',
 'de Messemaeker',
 'Chaffee',
 'Youseff',
 'Ilmakangas',
 'Sutton',
 'Turpin',
 'Cohen',
 'Kimball',
 'Aks',
 'Webber',
 'Kallio',
 'Yrois',
 'McKane',
 'Mineff',
 'Sjostedt',
 'Hirvonen',
 'Hood',
 'Slemen',
 'Cribb',
 'Beane',
 'McGowan',
 'Corn',
 'Brown',
 'Greenberg',
 'Elsbury',
 'Sivola',
 'Wheadon',
 'Chapman',


In [ ]:
gender_surv = titanic.groupby(["Survived", "Sex"]).count().rename(columns={"Name":"count"})["count"].reset_index()
pclass_surv = titanic.groupby(["Survived", "Pclass"]).count().rename(columns={"Name":"count"})["count"].reset_index()
embarked_surv = titanic.groupby(["Survived", "Embarked"]).count().rename(columns={"Name":"count"})["count"].reset_index()
embarked_surv = embarked_surv.replace({"C":"Cherbourg",
                                       "Q":"Queenstown",
                                       "S":"Southampton"})
age_survived = titanic[titanic["Survived"]==1]
age_perished = titanic[titanic["Survived"]==0]


fig_2 = make_subplots(rows=2, cols=2)

fig_2.add_trace(
    go.Bar(
        x=gender_surv["Sex"],
        y=gender_surv["count"],
        marker_color=["green" if val==1 else "red" for val in gender_surv["Survived"]],
        text=["survived" if val==1 else "perished" for val in gender_surv["Survived"]],
    ),
    row=1,
    col=1
)

fig_2.add_trace(
    go.Bar(
        x=pclass_surv["Pclass"],
        y=pclass_surv["count"],
        marker_color=["green" if val==1 else "red" for val in pclass_surv["Survived"]],
        text=["survived" if val==1 else "perished" for val in pclass_surv["Survived"]],
    ),
    row=1,
    col=2
)

fig_2.add_trace(
    go.Bar(
        x=embarked_surv["Embarked"],
        y=embarked_surv["count"],
        marker_color=["green" if val==1 else "red" for val in embarked_surv["Survived"]],
        text=["survived" if val==1 else "perished" for val in embarked_surv["Survived"]],
    ),
    row=2,
    col=1
)

fig_2.add_trace(
    go.Histogram(
        x=age_survived["Age"],
        marker=dict(opacity=0.75, color="green")
    ),
    row=2,
    col=2
)

fig_2.add_trace(
    go.Histogram(
        x=age_perished["Age"],
        marker=dict(opacity=0.75, color="red")
    ),
    row=2,
    col=2
)

fig_2.update_xaxes(title_text="gender", row=1, col=1)
fig_2.update_xaxes(title_text="passenger class", row=1, col=2)
fig_2.update_xaxes(title_text="port of embarkation", row=2, col=1)

fig_2.update_layout(showlegend=False, height=700, width=1000)

fig_2.show()

In [ ]:
fare_survived = titanic[titanic["Survived"]==1]
fare_perished = titanic[titanic["Survived"]==0]

fig_3 = make_subplots(rows=2, cols=2)

fig_3.add_trace(
    go.Scatter(
        x=titanic["Fare"],
        y=titanic["Pclass"],
        mode="markers"
    ),
    row=1,
    col=1
)

fig_3.show()

In [304]:
# titanic.groupby("LastName").count().sort_values("Name", ascending=False)
titanic[titanic["Fare"]==0]

,PassengerId,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass,Survived,FirstName,LastName,Salutation,FamilySize,Pclass*Age
143,180,"Leonard, Mr. Lionel",male,36.0,0,0,LINE,0.0,NaN,S,3,0,Mr. Lionel,Leonard,Mr.,0,108.0
211,264,"Harrison, Mr. William",male,40.0,0,0,112059,0.0,B94,S,1,0,Mr. William,Harrison,Mr.,0,40.0
217,272,"Tornquist, Mr. William Henry",male,25.0,0,0,LINE,0.0,NaN,S,3,1,Mr. William Henry,Tornquist,Mr.,0,75.0
241,303,"Johnson, Mr. William Cahoone Jr",male,19.0,0,0,LINE,0.0,NaN,S,3,0,Mr. William Cahoone Jr,Johnson,Mr.,0,57.0
471,598,"Johnson, Mr. Alfred",male,49.0,0,0,LINE,0.0,NaN,S,3,0,Mr. Alfred,Johnson,Mr.,0,147.0
642,807,"Andrews, Mr. Thomas Jr",male,39.0,0,0,112050,0.0,A36,S,1,0,Mr. Thomas Jr,Andrews,Mr.,0,39.0
657,823,"Reuchlin, Jonkheer. John George",male,38.0,0,0,19972,0.0,NaN,S,1,0,Jonkheer. John George,Reuchlin,Jonkheer.,0,38.0


In [305]:
numerical_data = ["Age", "Fare", "Pclass", "FamilySize"]
num_X = titanic[numerical_data]

categorical_data = ["Sex"]
cat_X = titanic[categorical_data]

X = pd.concat([num_X, cat_X], axis=1)
y = titanic["Survived"]


##### Create Encoder for categorical data; 
#####   - fit on the entire dataset; transform on train and test set separately
ohe_enc = OneHotEncoder(sparse_output=False)
ohe_enc.fit(cat_X)

##### Create Scaler for numerical data;
#####   - fit on the entire dataset; transform on train and test set separately
max_scaler = MaxAbsScaler()
max_scaler.fit(num_X)

std_scaler = StandardScaler()
std_scaler.fit(num_X)

##### Split data into train set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=43)

##### Split data into numerical and categorical data
cat_X_train =  X_train[categorical_data].reset_index(drop=True)
num_X_train = X_train.drop(columns= categorical_data).reset_index(drop=True)

cat_X_test =  X_test[categorical_data].reset_index(drop=True)
num_X_test = X_test.drop(columns= categorical_data).reset_index(drop=True)

##### Transform categorical data using one-hot encoder
X_train_enc = ohe_enc.transform(cat_X_train)
X_train_enc = pd.DataFrame(X_train_enc, columns=ohe_enc.get_feature_names_out())

X_test_enc = ohe_enc.transform(cat_X_test)
X_test_enc = pd.DataFrame(X_test_enc, columns=ohe_enc.get_feature_names_out())

##### Transform numerical data using scaler
# X_train_scaled = max_scaler.transform(num_X_train)
X_train_scaled = std_scaler.transform(num_X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=max_scaler.feature_names_in_)

# X_test_scaled = max_scaler.transform(num_X_test)
X_test_scaled = std_scaler.transform(num_X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=max_scaler.feature_names_in_)

##### Recombine the numerical and categorical data
new_X_train = pd.concat([X_train_enc, X_train_scaled], axis=1)
new_X_test = pd.concat([X_test_enc, X_test_scaled], axis=1)

new_X_train

,Sex_female,Sex_male,Age,Fare,Pclass,FamilySize
0,0.0,1.0,0.300903,-0.407687,-0.287191,-0.637897
1,0.0,1.0,0.231856,-0.489836,0.908600,-0.637897
2,1.0,0.0,-0.182430,-0.503620,0.908600,-0.637897
3,0.0,1.0,-1.080050,0.233127,0.908600,4.079136
4,1.0,0.0,-1.977669,-0.442974,0.908600,0.709826
...,...,...,...,...,...,...
472,1.0,0.0,0.093760,-0.313172,0.908600,0.035965
473,0.0,1.0,-1.989407,-0.105238,-0.287191,0.709826
474,1.0,0.0,-0.389573,-0.407687,-0.287191,-0.637897
475,1.0,0.0,0.715189,1.889036,-1.482983,0.709826


In [306]:
neighbor_count = [val for val in range(1,21)]
accuracy_list = []
precision_list = []
f1_score_list = []

for val in neighbor_count:
    neigh_titanic = KNeighborsClassifier(n_neighbors=val)
    neigh_titanic.fit(new_X_train, y_train)

    titanic_prediction = neigh_titanic.predict(new_X_test)

    accuracy = accuracy_score(y_test, titanic_prediction)
    accuracy_list.append(accuracy)

    precision = average_precision_score(y_test, titanic_prediction)
    precision_list.append(precision)

    f1 = f1_score(y_test, titanic_prediction)
    f1_score_list.append(f1)


metric_list = ["accuracy", "precision", "f1_score"]
column_names = ["n_neighbor"] + metric_list

prediction_scores = pd.DataFrame(list(zip(neighbor_count, accuracy_list, precision_list, f1_score_list)), columns=column_names)

fig_4 = go.Figure()

for val in metric_list:

    fig_4.add_trace(
        go.Scatter(
            x=neighbor_count,
            y=prediction_scores[val],
            name=val
        )
    )

fig_4.update_xaxes(title_text="n_neighbors")
fig_4.update_yaxes(title_text="score value")
fig_4.show()

---
# <center> Prediction </center>
----

In [347]:
titanic_test = pd.read_csv("titanic_test.csv")

################################################# Data Clean up ####################################################
##### Cannot remove any row with NaN; Kaggle will not accept it

##### replace NaN Age with mean Age based on gender
gender_mean_age = titanic_test[["Age", "Sex", "Pclass"]].groupby("Sex").mean()
titanic_test.loc[(titanic_test["Age"].isna()) & (titanic_test["Sex"]=="male"), "Age"] = gender_mean_age.loc["male", "Age"]
titanic_test.loc[(titanic_test["Age"].isna()) & (titanic_test["Sex"]=="female"), "Age"] = gender_mean_age.loc["female", "Age"]


##### replace NaN Fare with average Fare from Pclass = 3 since the one case is from this Pclass
mean_fare = titanic_test.loc[titanic_test["Pclass"]==3, "Fare"].mean()
titanic_test.loc[titanic_test["Fare"].isna(), "Fare"] = mean_fare


################################################# Feature Engineering ##############################################

##### split first and last name
titanic_test["FirstName"] = [titanic_test.loc[val, "Name"].split(",")[1] for val in range(len(titanic_test))]
titanic_test["LastName"] = [titanic_test.loc[val, "Name"].split(",")[0] for val in range(len(titanic_test))]

##### get salutation
titanic_test["Salutation"] = [titanic_test.loc[val, "FirstName"].split()[0] for val in range(len(titanic_test))]

##### determine family size
titanic_test["FamilySize"] = titanic_test["SibSp"] + titanic_test["Parch"]

##### Pclass-Age interaction
titanic_test["Pclass*Age"] = titanic_test["Pclass"]*titanic_test["Age"]

titanic_test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName,LastName,Salutation,FamilySize,Pclass*Age
0,892,3,"Kelly, Mr. James",male,34.500000,0,0,330911,7.8292,NaN,Q,Mr. James,Kelly,Mr.,0,103.500000
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.000000,1,0,363272,7.0000,NaN,S,Mrs. James (Ellen Needs),Wilkes,Mrs.,1,141.000000
2,894,2,"Myles, Mr. Thomas Francis",male,62.000000,0,0,240276,9.6875,NaN,Q,Mr. Thomas Francis,Myles,Mr.,0,124.000000
3,895,3,"Wirz, Mr. Albert",male,27.000000,0,0,315154,8.6625,NaN,S,Mr. Albert,Wirz,Mr.,0,81.000000
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.000000,1,1,3101298,12.2875,NaN,S,Mrs. Alexander (Helga E Lindqvist),Hirvonen,Mrs.,2,66.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,30.272732,0,0,A.5. 3236,8.0500,NaN,S,Mr. Woolf,Spector,Mr.,0,90.818195
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.000000,0,0,PC 17758,108.9000,C105,C,Dona. Fermina,Oliva y Ocana,Dona.,0,39.000000
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.500000,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,Mr. Simon Sivertsen,Saether,Mr.,0,115.500000
416,1308,3,"Ware, Mr. Frederick",male,30.272732,0,0,359309,8.0500,NaN,S,Mr. Frederick,Ware,Mr.,0,90.818195


In [348]:
##### Data treatment

##### Split data into numerical and categorical data
cat_titanic_test =  titanic_test[categorical_data].reset_index(drop=True)
num_titanic_test = titanic_test[numerical_data].reset_index(drop=True)


##### Transform categorical data using one-hot encoder
cat_titanic_test_enc = ohe_enc.transform(cat_titanic_test)
cat_titanic_test_enc = pd.DataFrame(cat_titanic_test_enc, columns=ohe_enc.get_feature_names_out())

##### Transform numerical data using scaler
# X_train_scaled = max_scaler.transform(num_X_train)
num_titanic_test_scaled = std_scaler.transform(num_titanic_test)
num_titanic_test_scaled = pd.DataFrame(num_titanic_test_scaled, columns=max_scaler.feature_names_in_)

##### Recombine the numerical and categorical data
new_titanic_test = pd.concat([cat_titanic_test_enc, num_titanic_test_scaled], axis=1)

new_titanic_test

,Sex_female,Sex_male,Age,Fare,Pclass,FamilySize
0,0.0,1.0,0.335427,-0.505431,0.908600,-0.637897
1,1.0,0.0,1.198523,-0.521106,0.908600,0.035965
2,0.0,1.0,2.234238,-0.470304,-0.287191,-0.637897
3,0.0,1.0,-0.182430,-0.489679,0.908600,-0.637897
4,1.0,0.0,-0.527669,-0.421156,0.908600,0.709826
...,...,...,...,...,...,...
413,0.0,1.0,0.043544,-0.501257,0.908600,-0.637897
414,1.0,0.0,0.646142,1.405117,-1.482983,-0.637897
415,0.0,1.0,0.611618,-0.516380,0.908600,-0.637897
416,0.0,1.0,0.043544,-0.501257,0.908600,-0.637897


In [349]:
titanic_test_prediction = neigh_titanic.predict(new_titanic_test)

submission = titanic_test
submission["Survived"] = titanic_test_prediction
submission = submission[["PassengerId", "Survived"]]

submission.to_csv("titanic_submission.csv", index=False)